# Dengue en Loreto (2000 - 2004)

## Notebook 01 - Exploración y calidad de datos

**Pregunta de análisis:** ¿Cuándo, dónde y a quiénes afecta más el dengue en Loreto, y qué cambió en las epidemias de 2023 y 2024?

**Fuente:** Vigilancia epidemiológica de dengue, Centro Nacional de Epidemiología, Prevencion y Control de Enfermedades (CDC Perú - MINSA), Plataforma Nacional de Datos Abiertos.

Este notebook carga el dataset nacional, lo compara con su diccionario de datos, diagnostica y corrige problemas de codificación de texto, valida los datos contra una cifra oficial y filtra los cados de Loreto.

**Contenido**
1. Lectura del archivo
2. Carga de datos
3. Columnas reales frente al diccionario
4. Exploración del dataset nacional
5. Codificación de texto: diagnóstico y corrección
6. Validación
7. Filtro de Loreto
8. Inventario de datos y limitaciones

**Resultado:** `data/processed/dengue_loreto.csv`

> ⚠️ Los datos crudos (103 MB) no se suben a GitHub porque superan el límite de 100 MB. Se descargan desde la Plataforma Nacional de Datos Abiertos y se guardan en `data/raw/`.

## 1. Lectura del archivo

Antes de cargar un archivo grande con pandas, se revisan sus primeras líneas en crudo para identificar su separador y posibles problemas de codificación.

In [2]:
import re
import pandas as pd

ruta = "../data/raw/datos_abiertos_vigilancia_dengue_2000_2024.csv"

with open(ruta, encoding="utf-8", errors="replace") as archivo:
    for _ in range(5):
        print(archivo.readline())

departamento;provincia;distrito;localidad;enfermedad;ano;semana;diagnostic;diresa;ubigeo;localcod;edad;tipo_edad;sexo

LORETO;ALTO AMAZONAS;YURIMAGUAS;000101;DENGUE SIN SIGNOS DE ALARMA;2000;1;A97.0;16;160201;;58;A;M

LORETO;ALTO AMAZONAS;YURIMAGUAS;000101;DENGUE SIN SIGNOS DE ALARMA;2000;1;A97.0;16;160201;;44;A;F

LORETO;ALTO AMAZONAS;YURIMAGUAS;BARRIO MORALILLOS;DENGUE SIN SIGNOS DE ALARMA;2000;1;A97.0;16;160201;;21;A;F

LORETO;MAYNAS;IQUITOS;005902;DENGUE SIN SIGNOS DE ALARMA;2000;1;A97.0;16;160101;;74;A;M



### 1.1 Detección automática de la codificación

Usamos `charset-normalizer`, analiza los bytes del archivoy estima su codificación. Se requiere instalarse en con el entorno activado: `pip install charset-normalizer`.

In [3]:
from charset_normalizer import from_path

deteccion = from_path(ruta).best()
print("Codificación detectada:", deteccion.encoding)

Codificación detectada: utf_8


**Hallazgos de la lectura**
- El separador es punto y coma (`;`), no coma.
- Las columnas reales incluyen `localidad` y `localcod`, que no figuran en el diccionario [Vigilancia Epidemiológica de Dengue](https://www.datosabiertos.gob.pe/dataset/vigilancia-epidemiol%C3%B3gica-de-dengue)

---
## 2. Carga de datos

Según el diccionario, `ubigeo`, `edad` y `diresa` son texto. Se indica explícitamente con `dtype` para que pandas no las convierta a número: los códigos perderían los ceros a la izquierda y la edad se mezclaría con su unidad (`tipo_edad`).

In [4]:
CODIFICACION = "utf-8" 

df = pd.read_csv(
    ruta,
    sep=";",
    encoding=CODIFICACION,
    dtype={"ubigeo": str, "edad": str, "diresa": str, "localcod": str}
)
df.shape

(1029421, 14)

In [5]:
# Usamos "deep" para medir tambien el texto que ocupa mucha memoria en pandas
df.info(memory_usage="deep")


<class 'pandas.DataFrame'>
RangeIndex: 1029421 entries, 0 to 1029420
Data columns (total 14 columns):
 #   Column        Non-Null Count    Dtype
---  ------        --------------    -----
 0   departamento  1029421 non-null  str  
 1   provincia     1029421 non-null  str  
 2   distrito      1029421 non-null  str  
 3   localidad     906823 non-null   str  
 4   enfermedad    1029421 non-null  str  
 5   ano           1029421 non-null  int64
 6   semana        1029421 non-null  int64
 7   diagnostic    1029421 non-null  str  
 8   diresa        1029398 non-null  str  
 9   ubigeo        1029421 non-null  str  
 10  localcod      880780 non-null   str  
 11  edad          1029421 non-null  str  
 12  tipo_edad     1029421 non-null  str  
 13  sexo          1029421 non-null  str  
dtypes: int64(2), str(12)
memory usage: 672.6 MB


In [6]:
df.head()

,departamento,provincia,distrito,localidad,enfermedad,ano,semana,diagnostic,diresa,ubigeo,localcod,edad,tipo_edad,sexo
0,LORETO,ALTO AMAZONAS,YURIMAGUAS,000101,DENGUE SIN SIGNOS DE ALARMA,2000,1,A97.0,16,160201,NaN,58,A,M
1,LORETO,ALTO AMAZONAS,YURIMAGUAS,000101,DENGUE SIN SIGNOS DE ALARMA,2000,1,A97.0,16,160201,NaN,44,A,F
2,LORETO,ALTO AMAZONAS,YURIMAGUAS,BARRIO MORALILLOS,DENGUE SIN SIGNOS DE ALARMA,2000,1,A97.0,16,160201,NaN,21,A,F
3,LORETO,MAYNAS,IQUITOS,005902,DENGUE SIN SIGNOS DE ALARMA,2000,1,A97.0,16,160101,NaN,74,A,M
4,LORETO,MAYNAS,IQUITOS,CARDOZO,DENGUE SIN SIGNOS DE ALARMA,2000,1,A97.0,16,160101,NaN,42,A,F


**Hallazgos**
- **1,029,421 filas**: cada fila es un caso individual de dengue (hay edad y sexo por fila).
- El archivo pesa 103 MB, pero en memoria ocupa unos 672.6 MB: el texto ocupa mucho más espacio en pandas que en el CSV. Por eso conviene filtrar Loreto pronto.

---
## 3. Columnas reales frente al diccionario

El diccionario de datos del portal describe 13 columnas. Se compara con las columnas reales del archivo.

In [7]:
columnas_diccionario = [
    "departamento", "provincia", "distrito", "enfermedad", "ano", "semana",
    "diagnostic", "tipo_dx", "diresa", "ubigeo", "edad", "tipo_edad", "sexo"
]

print("En el diccionario pero no en el archivo:", set(columnas_diccionario) - set(df.columns))
print("En el archivo pero no en el diccionario:", set(df.columns) - set(columnas_diccionario))

En el diccionario pero no en el archivo: {'tipo_dx'}
En el archivo pero no en el diccionario: {'localidad', 'localcod'}


**Hallazgo:** el diccionario y los datos no coinciden en las dos direcciones.
- **Falta `tipo_dx`**: no es posible distinguir casos confirmados, probables y sospechosos.
- **Sobran `localidad` y `localcod`**, que no están documentadas.

Cuando el diccionario y los datos no coinciden, mandan los datos.